# Human Memory Experiment Analysis

This notebook runs the shared analysis pipeline on all `data/final/final_*.csv` files. It writes standard figures and CSV summaries to `final_analysis/`.

The final sessions contain the repeated experiment used for the main analysis. Run the notebook from the `dtu-human-memory` directory.

## Load and Analyze

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import Image, display
import pandas as pd

if not Path("analysis_utils.py").exists():
    raise FileNotFoundError(
        "Run this notebook from the dtu-human-memory directory."
    )

import analysis_utils as analysis

DATA_DIR = Path("data") / "final"
FILE_PATTERN = "final_*.csv"
OUT_DIR = Path("final_analysis")
PAPER_FIGURES = [
    "figure_1_free_recall.png",
    "figure_2_primacy_recency.png",
    "figure_3_serial_capacity.png",
    "figure_4_serial_manipulations.png",
]
for filename in PAPER_FIGURES:
    (OUT_DIR / filename).unlink(missing_ok=True)

df = analysis.run_analysis(
    DATA_DIR,
    OUT_DIR,
    FILE_PATTERN,
    dataset_name="final experiment",
)
df = analysis.completed_trials(df)
df["accuracy"] = pd.to_numeric(df["accuracy"], errors="coerce")
df["whole_sequence_correct"] = pd.to_numeric(
    df["whole_sequence_correct"],
    errors="coerce",
)

print(f"Analyzing {len(df)} completed trials from {df['source_file'].nunique()} file(s).")
print(f"Participants: {df['participant_id'].nunique()}")

## Notebook Overview

This chart summarizes pooled trial accuracy by section. The detailed section analyses and planned contrasts are generated by the shared utility module.

In [ ]:
section_summary = (
    df.groupby("section")["accuracy"]
    .agg(mean="mean", n="size")
    .sort_values("mean")
)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(section_summary.index, section_summary["mean"], color="#386cb0")
ax.set_title("Final experiment mean accuracy by section")
ax.set_xlabel("Accuracy")
ax.set_xlim(0, 1.18)
ax.xaxis.set_major_formatter(PercentFormatter(1))
for bar, (_, row) in zip(bars, section_summary.iterrows()):
    ax.text(
        bar.get_width() + 0.02,
        bar.get_y() + bar.get_height() / 2,
        f"{row['mean']:.0%} (n={int(row['n'])})",
        va="center",
    )
fig.tight_layout()
fig.savefig(OUT_DIR / "notebook_overview.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)

## Paper Figures From This Run

In [ ]:
for filename in PAPER_FIGURES:
    path = OUT_DIR / filename
    if path.exists():
        print(f"{path.name} (generated in this run)")
        display(Image(filename=str(path)))
    else:
        print(f"{path.name}: not generated because the required section has no data")

## Exported Tables

The CSV files are the reproducible outputs used for reporting and further statistical modelling.

In [ ]:
for path in sorted(OUT_DIR.glob("*.csv")):
    print(f"## {path.name}")
    print(pd.read_csv(path).head(20).to_string(index=False))